# Visium Thymus Flash Analysis Pipeline

This notebook analyzes spatial transcriptomics data from a Visium Thymus Flash experiment.

- **Reusable functions** are imported from the `spatial_transcriptomics` package for configuration, data loading, QC, analysis, and plotting.
- **Analysis-specific decisions** remain documented in the notebook for this experiment.
- **Current scope:** load FD1 at 8 µm, attach spatial coordinates, and inspect unfiltered QC. Filtering, normalization, clustering, and downstream analysis are intentionally deferred.

## Experimental context and input requirements

For each tissue section, the Space Ranger output should provide:

- a filtered feature-barcode matrix;
- spatial coordinates and scale factors;
- tissue images;
- bin- or cell-level metadata.

The project sample sheet should eventually record at least `sample_id`, `mouse_id`, `condition` (`control`, `conv`, or `flash`), `timepoint`, the Space Ranger source path, and the selected bin size. Histology-informed ROI labels, section-quality notes, and a matching scRNA-seq thymus reference for later deconvolution are also recommended.

For the current FD1 pass, `binned_outputs.tar.gz` remains unchanged on Oak and only `square_008um` is extracted into job-local SCG scratch. The expected staged directory is `${TMPDIR}/FD1_HD/binned_outputs/square_008um/`, containing `filtered_feature_bc_matrix.h5` and `spatial/tissue_positions.parquet`. "Unfiltered QC" here means no additional repository-level filtering after Space Ranger's filtered matrix.

## 1. Setup and configuration

The default is the SCG configuration. Set `SPATIAL_CONFIG` before starting Jupyter to select a different config.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
from scipy import sparse

from spatial_transcriptomics.config import (
    get_config_section,
    get_section_path,
    load_config,
)
from spatial_transcriptomics.data import (
    calc_qc_metrics,
    load_reference_genes,
    load_visium_hd_bin,
)

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white")

config_path = os.environ.get("SPATIAL_CONFIG", "configs/cluster.yaml")
config = load_config(config_path)
repo_root = Path(str(config["repo_root"]))
hd_config = get_config_section(config, "visium_hd")
sample_id = str(hd_config.get("sample_id", "FD_1"))
bin_size_um = int(hd_config.get("bin_size_um", 8))
staged_dir = get_section_path(
    config, hd_config, "staged_dir", "${TMPDIR}/FD1_HD"
)
load_images = bool(hd_config.get("load_images", True))
use_filtered_matrix = bool(hd_config.get("use_filtered_matrix", True))

print(f"Config: {config_path}")
print(f"Sample: {sample_id}; bin size: {bin_size_um} µm")
print(f"Staged input: {staged_dir}")
print(f"TMPDIR: {os.environ.get('TMPDIR', '<not set>')}")

## 2. Validate the staged input

In [ ]:
bin_name = f"square_{bin_size_um:03d}um"
bin_dir = staged_dir / "binned_outputs" / bin_name
matrix_path = bin_dir / "filtered_feature_bc_matrix.h5"
positions_path = bin_dir / "spatial" / "tissue_positions.parquet"
required_paths = [bin_dir, matrix_path, positions_path]
missing = [path for path in required_paths if not path.exists()]
if missing:
    missing_text = "\n".join(f"- {path}" for path in missing)
    raise FileNotFoundError(
        "The staged Visium HD input is incomplete. Missing:\n" + missing_text
    )

print("Required 8 µm inputs found:")
for path in required_paths:
    print(f"- {path}")

## 3. Load matrix and spatial coordinates

The matrix remains sparse. No filtering, normalization, scaling, PCA, or clustering is performed.

In [ ]:
adata = load_visium_hd_bin(
    staged_dir,
    bin_size_um=bin_size_um,
    library_id=sample_id,
    load_images=load_images,
    use_filtered_matrix=use_filtered_matrix,
)

print(adata)
print(f"Bins: {adata.n_obs:,}")
print(f"Features: {adata.n_vars:,}")
print(f"Sparse count matrix: {sparse.issparse(adata.X)}")
print(f"Spatial coordinate shape: {adata.obsm['spatial'].shape}")
print(f"Spatial libraries: {list(adata.uns.get('spatial', {}))}")

## 4. Calculate unfiltered QC metrics

Metrics are calculated on the Space Ranger filtered matrix. No additional bins are removed. The repository's conventional-Visium thresholds are deliberately not applied.

In [ ]:
mt_genes = load_reference_genes(
    repo_root / "reference_genomes" / "mouse_mitochondrial_genes.txt"
)
rp_genes = load_reference_genes(
    repo_root / "reference_genomes" / "mouse_ribosomal_genes.txt"
)
adata = calc_qc_metrics(adata, mt_genes=mt_genes, rp_genes=rp_genes)

qc_columns = [
    column
    for column in [
        "total_counts",
        "n_genes_by_counts",
        "pct_counts_mt",
        "pct_counts_rp",
    ]
    if column in adata.obs
]
qc_summary = adata.obs[qc_columns].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T
qc_summary

## 5. Inspect QC distributions

In [ ]:
fig, axes = plt.subplots(1, len(qc_columns), figsize=(5 * len(qc_columns), 4))
if len(qc_columns) == 1:
    axes = [axes]

for ax, column in zip(axes, qc_columns):
    values = adata.obs[column].to_numpy()
    if column in {"total_counts", "n_genes_by_counts"}:
        values = np.log10(values + 1)
        xlabel = f"log10({column} + 1)"
    else:
        xlabel = column
    ax.hist(values, bins=80)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Number of 8 µm bins")

fig.suptitle(f"{sample_id}: unfiltered 8 µm QC distributions")
plt.tight_layout()
plt.show()

## 6. Inspect QC metrics in spatial coordinates

In [ ]:
coordinates = adata.obsm["spatial"]
spatial_metrics = ["total_counts", "n_genes_by_counts"]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, metric in zip(axes, spatial_metrics):
    values = np.log1p(adata.obs[metric].to_numpy())
    points = ax.scatter(
        coordinates[:, 0],
        coordinates[:, 1],
        c=values,
        cmap="viridis",
        s=1,
        linewidths=0,
        rasterized=True,
    )
    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.set_title(f"log1p({metric})")
    ax.set_axis_off()
    fig.colorbar(points, ax=ax, shrink=0.7)

fig.suptitle(f"{sample_id}: unfiltered spatial QC at {bin_size_um} µm")
plt.tight_layout()
plt.show()

## Stop here

Review the QC summary, distributions, and spatial patterns before selecting thresholds. Do not apply the repository's existing conventional-Visium filtering settings.